# Fase 1: Análisis y Exploración de Datos de Parkinson

En esta primera fase, vamos a:
1. Cargar y visualizar los datos.
2. Preprocesar las características numéricas (Scaling).
3. Dividir los datos en entrenamiento y validación/prueba.
4. Construir y entrenar un Perceptrón Multicapa (MLP) básico.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

## 1. Carga y Exploración de los Datos

In [ ]:
# Asegúrate de que el archivo 'parkinsons.data' esté en el mismo directorio
df = pd.read_csv('../Data_entrenamiento/Punto_1/parkinsons.data')
display(df.head())
print("\nInformación del dataset:")
df.info()

In [ ]:
# Verificamos la distribución de la clase objetivo (status)
# 1 = Parkinson, 0 = Sano
sns.countplot(x='status', data=df)
plt.title('Distribución de Clases (Parkinson vs Sanos)')
plt.show()

print(df['status'].value_counts())

## 2. Preprocesamiento de Datos
La columna `name` no aporta valor predictivo por lo que la eliminaremos. Además, escalaremos las variables ya que las redes neuronales son sensibles a la escala de las entradas.

In [ ]:
# Separar características (X) y objetivo (y)
X = df.drop(['name', 'status'], axis=1)
y = df['status']

# División de datos en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Dimensión X_train: {X_train.shape}')
print(f'Dimensión X_test: {X_test.shape}')

In [ ]:
# Escalamiento de datos (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Modelo de Red Neuronal (MLP Superficial)

In [ ]:
# Definición del modelo usando Keras
model = Sequential()

# Capa oculta con función de activación ReLU
# Usamos una arquitectura superficial: 1 capa oculta
model.add(Dense(32, input_dim=X_train_scaled.shape[1], activation='relu'))
model.add(Dropout(0.2)) # Prevención de sobreajuste

# Capa de salida con activación Sigmoid para clasificación binaria
model.add(Dense(1, activation='sigmoid'))

# Compilación del modelo
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

model.summary()

In [ ]:
# Entrenamiento del modelo
# Usamos el conjunto de prueba como datos de validación (validation_data) durante el entrenamiento
history = model.fit(X_train_scaled, y_train, 
                    epochs=50, 
                    batch_size=16, 
                    validation_data=(X_test_scaled, y_test), 
                    verbose=1)

## 4. Evaluación del Modelo

In [ ]:
# Graficar Loss y Accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Pérdida (Loss) vs Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Precisión (Accuracy) vs Epochs')
plt.legend()

plt.show()

In [ ]:
# Evaluación final y matriz de confusión
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Precisión final en el conjunto de prueba: {accuracy:.4f}\n")

# Predicciones
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))